SUPPLY CHAIN PERFORMANCE & RISK ANALYTICS 

In [1]:
# Import libraries 

import pandas as pd
import numpy as np

In [2]:
data_path = ("C:/Users/Admin/Downloads/Portfolio Projects/Supply Chain Performance and Risk Analytics/data/raw/DataCoSupplyChainDataset.csv")

In [3]:
# Load raw data 

print("\n1. LOAD DATA")

df = pd.read_csv(data_path, encoding = "latin-1")
print("\nDataset loaded successfully!") 


1. LOAD DATA

Dataset loaded successfully!


In [4]:
# Dataset Overview 

print("\n2. DATASET OVERVIEW")

print(f"Rows : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]:,}")

print("\nColumn Names : ")

for number, column in enumerate(df.columns, start = 1):
    print(f"{number :02d}. {column}") 


2. DATASET OVERVIEW
Rows : 180,519
Columns : 53

Column Names : 
01. Type
02. Days for shipping (real)
03. Days for shipment (scheduled)
04. Benefit per order
05. Sales per customer
06. Delivery Status
07. Late_delivery_risk
08. Category Id
09. Category Name
10. Customer City
11. Customer Country
12. Customer Email
13. Customer Fname
14. Customer Id
15. Customer Lname
16. Customer Password
17. Customer Segment
18. Customer State
19. Customer Street
20. Customer Zipcode
21. Department Id
22. Department Name
23. Latitude
24. Longitude
25. Market
26. Order City
27. Order Country
28. Order Customer Id
29. order date (DateOrders)
30. Order Id
31. Order Item Cardprod Id
32. Order Item Discount
33. Order Item Discount Rate
34. Order Item Id
35. Order Item Product Price
36. Order Item Profit Ratio
37. Order Item Quantity
38. Sales
39. Order Item Total
40. Order Profit Per Order
41. Order Region
42. Order State
43. Order Status
44. Order Zipcode
45. Product Card Id
46. Product Category Id
47. 

In [5]:
# Basic structure of dataset 

print("\n3. BASIC STRUCTURE")

print("\nFirst 5 records : ")
print(df.head())
print("\nLast 5 records : ")
print(df.tail())

print("\nData types : ", "\n", df.dtypes) 


3. BASIC STRUCTURE

First 5 records : 
       Type  Days for shipping (real)  Days for shipment (scheduled)  \
0     DEBIT                         3                              4   
1  TRANSFER                         5                              4   
2      CASH                         4                              4   
3     DEBIT                         3                              4   
4   PAYMENT                         2                              4   

   Benefit per order  Sales per customer   Delivery Status  \
0          91.250000          314.640015  Advance shipping   
1        -249.089996          311.359985     Late delivery   
2        -247.779999          309.720001  Shipping on time   
3          22.860001          304.809998  Advance shipping   
4         134.210007          298.250000  Advance shipping   

   Late_delivery_risk  Category Id   Category Name Customer City  ...  \
0                   0           73  Sporting Goods        Caguas  ...   
1       

In [6]:
# Missing value audit 

print("\n4. MISSING VALUE AUDIT")

missing_audit = pd.DataFrame({
    "Column" : df.columns, 
    "Missing Count" : df.isna().sum(), 
    "Missing %" : (df.isna().mean() * 100).round(4)
})

missing_audit = (missing_audit[missing_audit["Missing Count"] > 0]
                 .sort_values(by = "Missing Count", ascending = False))

print("\nColumns containing missing values : ")

if missing_audit.empty :
    print("No missing values detected.")
else :
    print(missing_audit.to_string(index = False)) 


4. MISSING VALUE AUDIT

Columns containing missing values : 
             Column  Missing Count  Missing %
Product Description         180519   100.0000
      Order Zipcode         155679    86.2397
     Customer Lname              8     0.0044
   Customer Zipcode              3     0.0017


In [7]:
# Duplicate Audit 

print("5. DUPLICATE AUDIT")

# Exact duplicate rows
exact_duplicates = df.duplicated().sum() 

print(f"\nExact duplicate rows : {exact_duplicates:,}") 

# Duplicate Order Item IDs
duplicate_order_items = df["Order Item Id"].duplicated().sum() 

print(f"Duplicate Order Item Ids : {duplicate_order_items:,}") 

# Duplicate Order IDs are expected because one order can contain multiple order items 
duplicate_orders = df["Order Id"].duplicated().sum()

print(f"Repeated Order Id rows : {duplicate_orders:,}") 

5. DUPLICATE AUDIT

Exact duplicate rows : 0
Duplicate Order Item Ids : 0
Repeated Order Id rows : 114,767


In [8]:
# Identifier Integrity Audit

print("6. IDENTIFIER INTEGRITY AUDIT")

identifier_columns = [
    "Order Id",
    "Order Item Id",
    "Order Customer Id",
    "Customer Id",
    "Product Card Id",
    "Order Item Cardprod Id",
    "Category Id",
    "Product Category Id",
    "Department Id"
]

identifier_audit = []

for column in identifier_columns : 

    identifier_audit.append({
        "Column": column,
        "Unique Values": df[column].nunique(dropna=True),
        "Missing Values": df[column].isna().sum(),
        "Minimum": df[column].min(),
        "Maximum": df[column].max()
    })

identifier_audit = pd.DataFrame(identifier_audit)

print(identifier_audit.to_string(index=False))

6. IDENTIFIER INTEGRITY AUDIT
                Column  Unique Values  Missing Values  Minimum  Maximum
              Order Id          65752               0        1    77204
         Order Item Id         180519               0        1   180519
     Order Customer Id          20652               0        1    20757
           Customer Id          20652               0        1    20757
       Product Card Id            118               0       19     1363
Order Item Cardprod Id            118               0       19     1363
           Category Id             51               0        2       76
   Product Category Id             51               0        2       76
         Department Id             11               0        2       12


In [9]:
print("\nCustomer ID consistency check:")

customer_id_mismatch = (df["Order Customer Id"] != df["Customer Id"]).sum()

print(f"Order Customer Id != Customer Id : " f"{customer_id_mismatch:,}") 


Customer ID consistency check:
Order Customer Id != Customer Id : 0


In [10]:
# 7. Date & Time Audit

print("7. DATE & TIME AUDIT")

date_columns = ["order date (DateOrders)", "shipping date (DateOrders)"]

for column in date_columns : 

    parsed_dates = pd.to_datetime(
        df[column],
        errors="coerce"
    )

    invalid_dates = parsed_dates.isna().sum()

    print(f"\n{column}")
    print(f"  Minimum date : {parsed_dates.min()}")
    print(f"  Maximum date : {parsed_dates.max()}")
    print(f"  Invalid dates: {invalid_dates:,}")

# Temporary datetime versions for validation
order_date = pd.to_datetime(
    df["order date (DateOrders)"],
    errors="coerce"
)

shipping_date = pd.to_datetime(
    df["shipping date (DateOrders)"],
    errors="coerce"
)

shipping_before_order = (
    shipping_date < order_date
).sum()

print(
    f"\nShipping date before order date : " 
    f"{shipping_before_order:,}"
)

7. DATE & TIME AUDIT

order date (DateOrders)
  Minimum date : 2015-01-01 00:00:00
  Maximum date : 2018-01-31 23:38:00
  Invalid dates: 0

shipping date (DateOrders)
  Minimum date : 2015-01-03 00:00:00
  Maximum date : 2018-02-06 22:14:00
  Invalid dates: 0

Shipping date before order date : 0


In [11]:
# Numerical Data Quality Audit

print("8. NUMERICAL DATA QUALITY AUDIT")

numeric_columns = df.select_dtypes(include=np.number).columns

numeric_audit = pd.DataFrame({
    "Column": numeric_columns,
    "Minimum": df[numeric_columns].min().values,
    "Maximum": df[numeric_columns].max().values,
    "Mean": df[numeric_columns].mean().values,
    "Missing": df[numeric_columns].isna().sum().values
})

print(numeric_audit.to_string(index=False))

8. NUMERICAL DATA QUALITY AUDIT
                       Column      Minimum       Maximum         Mean  Missing
     Days for shipping (real)     0.000000      6.000000     3.497654        0
Days for shipment (scheduled)     0.000000      4.000000     2.931847        0
            Benefit per order -4274.979980    911.799988    21.974989        0
           Sales per customer     7.490000   1939.989990   183.107609        0
           Late_delivery_risk     0.000000      1.000000     0.548291        0
                  Category Id     2.000000     76.000000    31.851451        0
                  Customer Id     1.000000  20757.000000  6691.379495        0
             Customer Zipcode   603.000000  99205.000000 35921.126914        3
                Department Id     2.000000     12.000000     5.443460        0
                     Latitude   -33.937553     48.781933    29.719955        0
                    Longitude  -158.025986    115.263077   -84.915675        0
            Order Cu

In [12]:
print("\nPotentially problematic numerical values:")

checks = {
    "Order Item Quantity": df["Order Item Quantity"] <= 0, 
    "Days for shipping (real)": df["Days for shipping (real)"] < 0, 
    "Days for shipment (scheduled)": df["Days for shipment (scheduled)"] < 0, 
    "Product Price": df["Product Price"] < 0, 
    "Order Item Product Price": df["Order Item Product Price"] < 0, 
    "Sales": df["Sales"] < 0, 
    "Order Item Total": df["Order Item Total"] < 0 
}

for column, condition in checks.items():
    print(
        f"{column:<35}: "     # left padding across 35 spaces
        f"{condition.sum():,}"
    ) 


Potentially problematic numerical values:
Order Item Quantity                : 0
Days for shipping (real)           : 0
Days for shipment (scheduled)      : 0
Product Price                      : 0
Order Item Product Price           : 0
Sales                              : 0
Order Item Total                   : 0


In [13]:
# CATEGORICAL DATA QUALITY AUDIT

print("9. CATEGORICAL DATA QUALITY AUDIT")

categorical_columns = [
    "Type", 
    "Delivery Status", 
    "Category Name", 
    "Customer Segment", 
    "Department Name", 
    "Market", 
    "Order Region", 
    "Order Status", 
    "Shipping Mode" 
]

for column in categorical_columns : 

    print(f"\n--- {column} ---")

    print(
        df[column]
        .value_counts(dropna=False)
        .to_string()
    ) 

9. CATEGORICAL DATA QUALITY AUDIT

--- Type ---
Type
DEBIT       69295
TRANSFER    49883
PAYMENT     41725
CASH        19616

--- Delivery Status ---
Delivery Status
Late delivery        98977
Advance shipping     41592
Shipping on time     32196
Shipping canceled     7754

--- Category Name ---
Category Name
Cleats                  24551
Men's Footwear          22246
Women's Apparel         21035
Indoor/Outdoor Games    19298
Fishing                 17325
Water Sports            15540
Camping & Hiking        13729
Cardio Equipment        12487
Shop By Sport           10984
Electronics              3156
Accessories              1780
Golf Balls               1475
Girls' Apparel           1201
Golf Gloves              1070
Trade-In                  974
Video Games               838
Children's Clothing       652
Women's Clothing          650
Baseball & Softball       632
Hockey                    614
Cameras                   592
Toys                      529
Golf Shoes                524

In [14]:
# DELIVERY PERFORMANCE AUDIT

print("10. DELIVERY PERFORMANCE AUDIT")

print("\nDelivery Status distribution :")

print(df["Delivery Status"].value_counts(dropna=False).to_string())

print("\nLate Delivery Risk distribution :")

print(df["Late_delivery_risk"].value_counts(dropna=False).to_string())

print("\nShipping Mode distribution :")

print(df["Shipping Mode"].value_counts(dropna=False).to_string())

10. DELIVERY PERFORMANCE AUDIT

Delivery Status distribution :
Delivery Status
Late delivery        98977
Advance shipping     41592
Shipping on time     32196
Shipping canceled     7754

Late Delivery Risk distribution :
Late_delivery_risk
1    98977
0    81542

Shipping Mode distribution :
Shipping Mode
Standard Class    107752
Second Class       35216
First Class        27814
Same Day            9737


In [15]:
shipping_variance = (
    df["Days for shipping (real)"]
    - df["Days for shipment (scheduled)"]
)

print("\nShipping Variance Summary:")

print(shipping_variance.describe().to_string())

print("\nShipping Variance Distribution:")

variance_group = pd.cut(
    shipping_variance,
    bins=[-np.inf, -0.01, 0.01, np.inf],
    labels=[
        "Faster than scheduled", 
        "On schedule", 
        "Slower than scheduled" 
    ]
)

print(variance_group.value_counts().to_string())


Shipping Variance Summary:
count    180519.000000
mean          0.565807
std           1.490966
min          -2.000000
25%           0.000000
50%           1.000000
75%           1.000000
max           4.000000

Shipping Variance Distribution:
Slower than scheduled    103400
Faster than scheduled     43366
On schedule               33753


In [16]:
print("11. CROSS-FIELD LOGIC CONSISTENCY AUDIT")

print("\nDelivery Status vs Shipping Variance:")

delivery_variance = pd.crosstab(
    df["Delivery Status"],
    variance_group,
    margins=True
)

print(delivery_variance.to_string())

print("\nLate Delivery Risk vs Shipping Variance:")

risk_variance = pd.crosstab(
    df["Late_delivery_risk"],
    variance_group,
    margins=True
)

print(risk_variance.to_string())

11. CROSS-FIELD LOGIC CONSISTENCY AUDIT

Delivery Status vs Shipping Variance:
col_0              Faster than scheduled  On schedule  Slower than scheduled     All
Delivery Status                                                                     
Advance shipping                   41592            0                      0   41592
Late delivery                          0            0                  98977   98977
Shipping canceled                   1774         1557                   4423    7754
Shipping on time                       0        32196                      0   32196
All                                43366        33753                 103400  180519

Late Delivery Risk vs Shipping Variance:
col_0               Faster than scheduled  On schedule  Slower than scheduled     All
Late_delivery_risk                                                                   
0                                   43366        33753                   4423   81542
1                         

In [17]:
print("\nDelivery Status vs Late Delivery Risk:")

status_risk = pd.crosstab(
    df["Delivery Status"], 
    df["Late_delivery_risk"], 
    margins=True 
)

print(status_risk.to_string())


Delivery Status vs Late Delivery Risk:
Late_delivery_risk      0      1     All
Delivery Status                         
Advance shipping    41592      0   41592
Late delivery           0  98977   98977
Shipping canceled    7754      0    7754
Shipping on time    32196      0   32196
All                 81542  98977  180519


In [18]:
# 12. FINANCIAL VALIDATION

print("12. FINANCIAL VALIDATION")

financial_columns = [
    "Sales", 
    "Sales per customer", 
    "Order Item Total", 
    "Order Item Discount", 
    "Order Item Discount Rate", 
    "Order Item Product Price", 
    "Order Profit Per Order", 
    "Order Item Profit Ratio", 
    "Benefit per order" 
]

financial_summary = pd.DataFrame({
    "Column": financial_columns, 
    "Total": [df[column].sum() for column in financial_columns], 
    "Mean": [df[column].mean() for column in financial_columns], 
    "Missing": [df[column].isna().sum() for column in financial_columns] 
})

print(financial_summary.to_string(index = False)) 

12. FINANCIAL VALIDATION
                  Column        Total       Mean  Missing
                   Sales 3.678474e+07 203.772096        0
      Sales per customer 3.305440e+07 183.107609        0
        Order Item Total 3.305440e+07 183.107609        0
     Order Item Discount 3.730378e+06  20.664741        0
Order Item Discount Rate 1.835304e+04   0.101668        0
Order Item Product Price 2.549516e+07 141.232550        0
  Order Profit Per Order 3.966903e+06  21.974989        0
 Order Item Profit Ratio 2.177901e+04   0.120647        0
       Benefit per order 3.966903e+06  21.974989        0


In [19]:
print("\nOrder Item Total reconciliation :")

calculated_total = (df["Order Item Product Price"] * df["Order Item Quantity"])

difference = (df["Order Item Total"] - calculated_total)

print(f"Mean difference : {difference.mean():.4f}")

print(
    f"Max absolute difference : "
    f"{difference.abs().max():.4f}"
)

print(
    f"Rows with difference > 0.01 : "
    f"{(difference.abs() > 0.01).sum():,}"
)


Order Item Total reconciliation :
Mean difference : -20.6645
Max absolute difference : 500.0000
Rows with difference > 0.01 : 170,491


In [30]:
# 12.1. FINANCIAL RECONCILIATION CHECK

print("12.1. FINANCIAL RECONCILIATION CHECK")

# Check 1: Product Price × Quantity

gross_item_value = (df["Order Item Product Price"] * df["Order Item Quantity"])

print("\n1. Gross Item Value")
print(
    f"Mean difference between Gross Item Value and Order Item Total: "
    f"{(gross_item_value - df['Order Item Total']).mean():.6f}"
)

# Check 2: Gross Item Value - Discount

calculated_item_total = (gross_item_value - df["Order Item Discount"])

difference = (calculated_item_total - df["Order Item Total"])

print("\n2. Discount Reconciliation")

print(f"Mean difference : {difference.mean():.6f}")

print(
    f"Maximum absolute diff : "
    f"{difference.abs().max():.6f}"
)

print(
    f"Rows within tolerance : "
    f"{(difference.abs() <= 0.01).sum():,}"
)

print(
    f"Rows outside tolerance: "
    f"{(difference.abs() > 0.01).sum():,}"
)


# Check 3: Sales vs Order Item Total

sales_difference = (df["Sales"] - df["Order Item Total"])

print("\n3. Sales vs Order Item Total")

print(
    f"Mean difference : "
    f"{sales_difference.mean():.6f}"
)

print(
    f"Maximum absolute diff : "
    f"{sales_difference.abs().max():.6f}"
)

print(
    f"Rows with difference > 0.01 : "
    f"{(sales_difference.abs() > 0.01).sum():,}"
)


# Check 4: Discount Rate

calculated_discount_rate = (df["Order Item Discount"] / gross_item_value)

discount_rate_difference = (calculated_discount_rate - df["Order Item Discount Rate"])

print("\n4. Discount Rate Reconciliation")

print(
    f"Mean difference : "
    f"{discount_rate_difference.mean():.8f}"
)

print(
    f"Maximum absolute diff : "
    f"{discount_rate_difference.abs().max():.8f}"
)

print(
    f"Rows within tolerance : "
    f"{(discount_rate_difference.abs() <= 0.0001).sum():,}"
)

12.1. FINANCIAL RECONCILIATION CHECK

1. Gross Item Value
Mean difference between Gross Item Value and Order Item Total: 20.664487

2. Discount Reconciliation
Mean difference : -0.000254
Maximum absolute diff : 0.010025
Rows within tolerance : 179,073
Rows outside tolerance: 1,446

3. Sales vs Order Item Total
Mean difference : 20.664488
Maximum absolute diff : 500.000000
Rows with difference > 0.01 : 170,491

4. Discount Rate Reconciliation
Mean difference : -0.00027070
Maximum absolute diff : 0.00540728
Rows within tolerance : 169,024


In [34]:
# 13. DATA AUDIT SUMMARY

print("\n" + "_" * 60)
print("\n13. DATA AUDIT SUMMARY")
print("_" * 60)

print("""
SUPPLY CHAIN PERFORMANCE & RISK ANALYTICS
DATA AUDIT FINDINGS
""")

print("-" * 60)
print("1. DATASET OVERVIEW")

print(f"Records analyzed     : {len(df):,}")
print(f"Columns analyzed     : {len(df.columns):,}")
print(f"Unique orders        : {df['Order Id'].nunique():,}")
print(f"Unique order items   : {df['Order Item Id'].nunique():,}")
print(f"Analytical grain     : One row per order item")
print(f"Exact duplicate rows : {df.duplicated().sum():,}")

print("-" * 60)
print("2. DATA COMPLETENESS")

print(f"Columns containing missing data : {df.isna().any().sum()}")

for column in missing_audit["Column"] :
    missing_count = df[column].isna().sum()
    missing_percent = df[column].isna().mean() * 100

    print(
        f"{column:<30}: "
        f"{missing_count:>8,} missing "
        f"({missing_percent:.2f}%)"
    )

print("-" * 60)
print("3. DATA INTEGRITY")

print(f"Duplicate Order Item Ids       : {duplicate_order_items:,}")
print(f"Order Customer ID mismatches   : {customer_id_mismatch:,}")
print(f"Invalid order dates            : 0")
print(f"Invalid shipping dates         : 0")
print(f"Shipping before order date     : {shipping_before_order:,}")

print("\nIdentifier relationships :")
print("Order Customer Id = Customer Id : VALID")
print("Product Card Id = Order Item Cardprod Id : VALID")
print("Category Id = Product Category Id : VALID")

print("-" * 60)
print("4. DELIVERY PERFORMANCE")

print(
    f"Late delivery records  : "
    f"{(df['Delivery Status'] == 'Late delivery').sum():,}"
)

print(
    f"Late delivery rate     : "
    f"{(df['Delivery Status'] == 'Late delivery').mean() * 100:.2f}%"
)

print(
    f"Mean shipping variance : "
    f"{shipping_variance.mean():.2f} days"
)

print(
    f"Slower than scheduled  : "
    f"{(shipping_variance > 0).sum():,}"
)

print(
    f"On schedule            : "
    f"{(shipping_variance == 0).sum():,}"
)

print(
    f"Faster than scheduled  : "
    f"{(shipping_variance < 0).sum():,}"
)

print("-" * 60)
print("5. FINANCIAL DATA QUALITY")

print(
    f"Negative profit records      : "
    f"{(df['Order Profit Per Order'] < 0).sum():,}"
)

print(
    f"Negative sales records       : "
    f"{(df['Sales'] < 0).sum():,}"
)

print(
    f"Negative order-total records : "
    f"{(df['Order Item Total'] < 0).sum():,}"
)

print("""\nFinancial reconciliation :
Product Price × Quantity does not equal Order Item Total directly.
After subtracting Order Item Discount, the relationship reconciles closely, with minor rounding differences.
Sales and Order Item Total represent different monetary measures 
and will therefore be retained as separate fields.""")

print("-" * 60)
print("6. OUTCOME / LEAKAGE FINDINGS")

print("""Delivery Status and Late_delivery_risk are outcome-related fields.
Late_delivery_risk corresponds directly to Late delivery records.
Shipping variance is calculated from actual vs scheduled shipping duration.

These fields will be treated as historical outcome/performance measures 
and will NOT be used as independent explanatory variables
if predictive analysis is introduced later.""")

print("-" * 60)
print("7. CLEANING DECISIONS")

print("""KEEP : 
- Transaction, order and product identifiers
- Order and shipping dates
- Shipping duration fields
- Shipping Mode
- Market and geographic fields
- Customer Segment
- Product / category / department information
- Sales, discounts and profit measures
- Order and delivery status for historical analysis""")

print("""\nEXCLUDE FROM ANALYTICAL LAYER : 
Product Description — 100% missing
Customer Email
Customer Fname
Customer Lname
Customer Password
Customer Street
Product Image""")

print("\nREVIEW / LIMIT USE:")
print("  • Order Zipcode — 86.24% missing")
print("  • Customer Zipcode — minimal missingness but not required")
print("  • Product Status — constant value")
print("  • Redundant identifier columns")

print("""\nDO NOT DELETE :
Negative profit records
Repeated Order Id rows
Shipping outcome records""")

print("-" * 60)
print("8. AUDIT CONCLUSION")

print("""
The dataset is structurally suitable for supply-chain performance
and operational-risk analysis.

No exact duplicate records were identified, order-item identifiers
are unique, date fields are valid, and key identifier relationships
are consistent.

The primary data-quality considerations are missing geographic
fields, redundant identifiers, outcome-related delivery fields,
and the distinction between gross sales and discounted order value.

The raw dataset will remain unchanged.

Cleaning will be performed in a separate analytical dataset,
with documented field-selection and transformation decisions.
""")

print("-" * 60)
print("DATA AUDIT COMPLETE")


____________________________________________________________

13. DATA AUDIT SUMMARY
____________________________________________________________

SUPPLY CHAIN PERFORMANCE & RISK ANALYTICS
DATA AUDIT FINDINGS

------------------------------------------------------------
1. DATASET OVERVIEW
Records analyzed     : 180,519
Columns analyzed     : 53
Unique orders        : 65,752
Unique order items   : 180,519
Analytical grain     : One row per order item
Exact duplicate rows : 0
------------------------------------------------------------
2. DATA COMPLETENESS
Columns containing missing data : 4
Product Description           :  180,519 missing (100.00%)
Order Zipcode                 :  155,679 missing (86.24%)
Customer Lname                :        8 missing (0.00%)
Customer Zipcode              :        3 missing (0.00%)
------------------------------------------------------------
3. DATA INTEGRITY
Duplicate Order Item Ids       : 0
Order Customer ID mismatches   : 0
Invalid order dates  